# Modelo Causal v3 — Variáveis Operacionais da Jornada do Pedido

No notebook anterior (causal_model_02), mostramos que o percentual de frete tem efeito causal praticamente nulo sobre OSR.

Neste notebook, analisamos variáveis **operacionais** da jornada do pedido:

*******PRECISO REVER ISSO ********

| Tratamento | Definição |
|---|---|
| `aprovacao_lenta` | Aprovação do pedido levou mais de 24h |
| `despacho_lento` | Despacho para transportadora levou mais de 3 dias |

| Outcome | Definição |
|---|---|
| `cancelado` | Pedido com status cancelado ou indisponível |
| `entrega_atrasada` | Entregue após a data estimada |
| `OSR` | Pedido entregue com sucesso |

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

from app.config.settings import INTERIM_DATA_DIR
from dowhy import CausalModel

import warnings
warnings.filterwarnings('ignore')

/Users/andreza/Documents/GitHub/TCC-CEDS-CAUSAL-AI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, "interim_dataset.parquet"))

print(df.shape)
df.head()

(97712, 34)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,avg_width,total_payment,avg_payment,max_installments,n_payments_type,OSR,installment_value,purchase_hour,purchase_weekday,purchase_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,13.0,38.71,12.903333,1.0,3.0,1,38.710000,10,0,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,19.0,141.46,141.460000,1.0,1.0,1,141.460000,20,1,7
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,21.0,179.12,179.120000,3.0,1.0,1,59.706667,8,2,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,20.0,72.20,72.200000,1.0,1.0,1,72.200000,19,5,11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,15.0,28.62,28.620000,1.0,1.0,1,28.620000,21,1,2


## 1. Criação de variáveis operacionais

A partir das datas da jornada do pedido, criamos:
- **Tempo de aprovação**: horas entre compra e aprovação do pedido
- **Tempo de despacho**: dias entre aprovação e envio à transportadora
- **Atraso na entrega**: dias de diferença entre entrega real e estimada

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Variáveis operacionais contínuas
df["approval_time_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df["dispatch_time_days"] = (
    df["order_delivered_carrier_date"] - df["order_approved_at"]
).dt.days

df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

# Tratamentos binários
df["aprovacao_lenta"] = (df["approval_time_hours"] > 24).astype(int)
df["despacho_lento"]  = (df["dispatch_time_days"] > 3).astype(int)

# Outcomes
df["cancelado"]        = (df["order_status"].isin(["canceled", "unavailable"])).astype(int)
df["entrega_atrasada"] = (df["delay_days"] > 0).astype(int)

print("=== Distribuição dos tratamentos ===")
print("aprovacao_lenta:\n", df["aprovacao_lenta"].value_counts(normalize=True).round(3))
print("\ndespacho_lento:\n", df["despacho_lento"].value_counts(normalize=True).round(3))

print("\n=== Distribuição dos outcomes ===")
print("cancelado:\n", df["cancelado"].value_counts(normalize=True).round(3))
print("\nentrega_atrasada:\n", df["entrega_atrasada"].value_counts(normalize=True).round(3))

=== Distribuição dos tratamentos ===
aprovacao_lenta:
 aprovacao_lenta
0    0.825
1    0.175
Name: proportion, dtype: float64

despacho_lento:
 despacho_lento
0    0.793
1    0.207
Name: proportion, dtype: float64

=== Distribuição dos outcomes ===
cancelado:
 cancelado
0    0.987
1    0.013
Name: proportion, dtype: float64

entrega_atrasada:
 entrega_atrasada
0    0.933
1    0.067
Name: proportion, dtype: float64


## 2. Confundidores (common causes)

Variáveis que influenciam tanto o tratamento quanto o outcome — precisam ser controladas no DAG.

In [6]:
common_causes = [
    "total_price",
    "n_items",
    "avg_weight",
    "customer_state",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "n_items_missing_info"
]

## 3. Análise 1 — Efeito de aprovação lenta sobre cancelamento

**Hipótese**: pedidos que demoram mais de 24h para serem aprovados têm maior probabilidade de serem cancelados.

In [7]:
cols_needed = common_causes + ["aprovacao_lenta", "cancelado"]
df_a1 = df[cols_needed].dropna().copy()

# Pré-processamento
le = LabelEncoder()
df_a1["customer_state"] = le.fit_transform(df_a1["customer_state"])

continuous_cols = [c for c in common_causes if c != "customer_state"]
scaler = StandardScaler()
df_a1[continuous_cols] = scaler.fit_transform(df_a1[continuous_cols])

print(f"Shape: {df_a1.shape}")
print(f"\nTaxa de cancelamento por aprovacao_lenta:")
print(df_a1.groupby("aprovacao_lenta")["cancelado"].mean().round(4))

Shape: (96929, 10)

Taxa de cancelamento por aprovacao_lenta:
aprovacao_lenta
0    0.0049
1    0.0046
Name: cancelado, dtype: float64


In [8]:
graph_a1 = """
digraph {
    aprovacao_lenta -> cancelado;

    total_price -> aprovacao_lenta;
    total_price -> cancelado;

    n_items -> aprovacao_lenta;
    n_items -> cancelado;

    avg_weight -> aprovacao_lenta;
    avg_weight -> cancelado;

    customer_state -> aprovacao_lenta;
    customer_state -> cancelado;

    purchase_month -> aprovacao_lenta;
    purchase_month -> cancelado;

    purchase_weekday -> aprovacao_lenta;
    purchase_weekday -> cancelado;

    purchase_hour -> aprovacao_lenta;
    purchase_hour -> cancelado;

    n_items_missing_info -> aprovacao_lenta;
    n_items_missing_info -> cancelado;
}
"""

model_a1 = CausalModel(
    data=df_a1,
    treatment="aprovacao_lenta",
    outcome="cancelado",
    graph=graph_a1
)

estimand_a1 = model_a1.identify_effect()
estimate_a1 = model_a1.estimate_effect(
    estimand_a1,
    method_name="backdoor.linear_regression"
)

print(f"ATE (aprovacao_lenta → cancelado): {estimate_a1.value:.6f}")

ATE (aprovacao_lenta → cancelado): -0.000274


In [9]:
# Refutação
r_placebo_a1 = model_a1.refute_estimate(estimand_a1, estimate_a1,
    method_name="placebo_treatment_refuter", num_simulations=100)

r_random_a1 = model_a1.refute_estimate(estimand_a1, estimate_a1,
    method_name="random_common_cause")

r_subset_a1 = model_a1.refute_estimate(estimand_a1, estimate_a1,
    method_name="data_subset_refuter", subset_fraction=0.8)

print("=== Placebo ===")
print(r_placebo_a1)
print("\n=== Random Common Cause ===")
print(r_random_a1)
print("\n=== Data Subset (80%) ===")
print(r_subset_a1)

=== Placebo ===
Refute: Use a Placebo Treatment
Estimated effect:-0.0002736318726485479
New effect:-3.519533921653042e-05
p value:0.94


=== Random Common Cause ===
Refute: Add a random common cause
Estimated effect:-0.0002736318726485479
New effect:-0.0002735796881486455
p value:0.94


=== Data Subset (80%) ===
Refute: Use a subset of data
Estimated effect:-0.0002736318726485479
New effect:-0.00033029771822896373
p value:0.86



## Análise 1 — Aprovação lenta → Cancelamento
|    | Valor | 
|---|---|
| ATE | -0.000274|
| Taxa atraso T=0 | 0.49%| 
| Taxa atraso T=1 | 0.46% | 

Efeito praticamente nulo — aprovação lenta não causa cancelamento. Acredito que faz sentido: o cliente cancela por outros motivos (mudou de ideia, preço, etc.), não por ter esperado pela aprovação e normalmente a aprovação costuma ser bem rápiada

Refutações: todas passaram (p ≥ 0.86) 

## 4. Análise 2 — Efeito de despacho lento sobre entrega atrasada

**Hipótese**: pedidos que demoram mais de 3 dias para serem despachados à transportadora têm maior probabilidade de chegar com atraso.

In [11]:
cols_needed2 = common_causes + ["despacho_lento", "entrega_atrasada"]
df_a2 = df[cols_needed2].dropna().copy()

# Pré-processamento
le2 = LabelEncoder()
df_a2["customer_state"] = le2.fit_transform(df_a2["customer_state"])

scaler2 = StandardScaler()
df_a2[continuous_cols] = scaler2.fit_transform(df_a2[continuous_cols])

print(f"Shape: {df_a2.shape}")
print(f"\nTaxa de atraso por despacho_lento:")
print(df_a2.groupby("despacho_lento")["entrega_atrasada"].mean().round(4))

Shape: (96929, 10)

Taxa de atraso por despacho_lento:
despacho_lento
0    0.0510
1    0.1297
Name: entrega_atrasada, dtype: float64


In [12]:
graph_a2 = """
digraph {
    despacho_lento -> entrega_atrasada;

    total_price -> despacho_lento;
    total_price -> entrega_atrasada;

    n_items -> despacho_lento;
    n_items -> entrega_atrasada;

    avg_weight -> despacho_lento;
    avg_weight -> entrega_atrasada;

    customer_state -> despacho_lento;
    customer_state -> entrega_atrasada;

    purchase_month -> despacho_lento;
    purchase_month -> entrega_atrasada;

    purchase_weekday -> despacho_lento;
    purchase_weekday -> entrega_atrasada;

    purchase_hour -> despacho_lento;
    purchase_hour -> entrega_atrasada;

    n_items_missing_info -> despacho_lento;
    n_items_missing_info -> entrega_atrasada;
}
"""

model_a2 = CausalModel(
    data=df_a2,
    treatment="despacho_lento",
    outcome="entrega_atrasada",
    graph=graph_a2
)

estimand_a2 = model_a2.identify_effect()
estimate_a2 = model_a2.estimate_effect(
    estimand_a2,
    method_name="backdoor.linear_regression"
)

print(f"ATE (despacho_lento → entrega_atrasada): {estimate_a2.value:.6f}")

ATE (despacho_lento → entrega_atrasada): 0.078961


In [13]:
# Refutação
r_placebo_a2 = model_a2.refute_estimate(estimand_a2, estimate_a2,
    method_name="placebo_treatment_refuter", num_simulations=100)

r_random_a2 = model_a2.refute_estimate(estimand_a2, estimate_a2,
    method_name="random_common_cause")

r_subset_a2 = model_a2.refute_estimate(estimand_a2, estimate_a2,
    method_name="data_subset_refuter", subset_fraction=0.8)

print("=== Placebo ===")
print(r_placebo_a2)
print("\n=== Random Common Cause ===")
print(r_random_a2)
print("\n=== Data Subset (80%) ===")
print(r_subset_a2)

=== Placebo ===
Refute: Use a Placebo Treatment
Estimated effect:0.07896103205277456
New effect:-0.00011030865931337855
p value:0.86


=== Random Common Cause ===
Refute: Add a random common cause
Estimated effect:0.07896103205277456
New effect:0.0789619139723089
p value:0.8600000000000001


=== Data Subset (80%) ===
Refute: Use a subset of data
Estimated effect:0.07896103205277456
New effect:0.07904288418587217
p value:0.8999999999999999



## Análise 2 — Despacho lento → Entrega atrasada
|    | Valor | 
|---|---|
| ATE | +0.079|
| Taxa atraso T=0 | 5.1% | 
| Taxa atraso T=1 | 12.9% | 

Efeito causal forte e claro: despacho lento (>3 dias) aumenta em 7.9 pontos percentuais a probabilidade de atraso na entrega. Pedidos com despacho lento têm 2.5x mais chance de atrasar.

Refutações: todas passaram (p ≥ 0.86) 

## 5. Resumo comparativo dos efeitos causais estimados

| Tratamento | Outcome | ATE | Interpretação |
|---|---|---|---|
| perc_freight < 5% | OSR | -0.000085 | Efeito nulo |
| aprovacao_lenta (>24h) | cancelado | -0.000274 | Efeito nulo |
| despacho_lento (>3d) | entrega_atrasada | Efeito Alto | |

Ou seja o tempo de despacho é a variavel operacional com maior efeito causal idendificado até agora. 